<h2>We basically have two main endpoints:</h2>

- r5ix-sfxw is health deficiencies, where every row is a different deficiency https://data.cms.gov/provider-data/dataset/r5ix-sfxw#api
- 4pq5-n9py is the provider information, where every row is a different facility https://data.cms.gov/provider-data/dataset/4pq5-n9py#api


In [266]:
import requests
import pandas as pd
import os
import json

<h2>First, let's check how many citations facilities get</h2>

In [267]:
url = "https://data.cms.gov/provider-data/api/1/datastore/query"

body = {
    "resource_id": "r5ix-sfxw",
    "limit": 1
}

# The URL comes from here: https://data.cms.gov/provider-data/dataset/r5ix-sfxw#api

r = requests.get("https://data.cms.gov/provider-data/api/1/datastore/query/r5ix-sfxw/0?limit=1")
print(r.status_code)
print(r.json())

# This doesn't work because we hit the limit of 1500

200
{'results': [{'cms_certification_number_ccn': '015009', 'provider_name': 'BURNS NURSING HOME, INC.', 'provider_address': '701 MONROE STREET NW', 'citytown': 'RUSSELLVILLE', 'state': 'AL', 'zip_code': '35653', 'survey_date': '2023-03-02', 'survey_type': 'Health', 'deficiency_prefix': 'F', 'deficiency_category': 'Resident Assessment and Care Planning Deficiencies', 'deficiency_tag_number': '0656', 'deficiency_description': "Develop and implement a complete care plan that meets all the resident's needs, with timetables and actions that can be measured.", 'scope_severity_code': 'J', 'deficiency_corrected': 'Past Non-Compliance', 'correction_date': '2023-01-13', 'inspection_cycle': '1', 'standard_deficiency': 'Y', 'complaint_deficiency': 'N', 'infection_control_inspection_deficiency': 'N', 'citation_under_idr': 'N', 'citation_under_iidr': 'N', 'location': '701 MONROE STREET NW,RUSSELLVILLE,AL,35653', 'processing_date': '2026-06-01'}], 'count': 418479, 'schema': {'327a1777-6c39-5872-9874

In [268]:
df = pd.DataFrame(all_df['results'])
df.head()

,cms_certification_number_ccn,provider_name,provider_address,citytown,state,zip_code,survey_date,survey_type,deficiency_prefix,deficiency_category,...,deficiency_corrected,correction_date,inspection_cycle,standard_deficiency,complaint_deficiency,infection_control_inspection_deficiency,citation_under_idr,citation_under_iidr,location,processing_date
0,015009,"BURNS NURSING HOME, INC.",701 MONROE STREET NW,RUSSELLVILLE,AL,35653,2023-03-02,Health,F,Resident Assessment and Care Planning Deficien...,...,Past Non-Compliance,2023-01-13,1,Y,N,N,N,N,"701 MONROE STREET NW,RUSSELLVILLE,AL,35653",2026-06-01


<h2> Build full DF </h2>
This works, but we have the limit to 1. We have to create a for loop for the entire dataframe. And we have to use a while true because there is a 1500 limit

In [273]:
all_fl = []
offset = 0

while True:
    r = requests.get(
        f"https://data.cms.gov/provider-data/api/1/datastore/query/r5ix-sfxw/0"
        f"?conditions[0][property]=state&conditions[0][value]=FL&conditions[0][operator]==&limit=1500&offset={offset}"
    )
    results = r.json().get('results', [])
    all_fl.extend(results)
    print(f"Offset {offset}: {len(results)} citations fetched")
    if len(results) < 1500:
        break
    offset += 1500

fl_df = pd.DataFrame(all_fl)
print(f"\nTotal FL citations: {len(fl_df)}")
print(f"Total FL facilities: {fl_df['cms_certification_number_ccn'].nunique()}")

Offset 0: 1500 citations fetched
Offset 1500: 1500 citations fetched
Offset 3000: 1500 citations fetched
Offset 4500: 1500 citations fetched
Offset 6000: 1500 citations fetched
Offset 7500: 1500 citations fetched
Offset 9000: 1500 citations fetched
Offset 10500: 1500 citations fetched
Offset 12000: 1500 citations fetched
Offset 13500: 81 citations fetched

Total FL citations: 13581
Total FL facilities: 688


In [274]:
print(fl_df.columns.tolist())

['cms_certification_number_ccn', 'provider_name', 'provider_address', 'citytown', 'state', 'zip_code', 'survey_date', 'survey_type', 'deficiency_prefix', 'deficiency_category', 'deficiency_tag_number', 'deficiency_description', 'scope_severity_code', 'deficiency_corrected', 'correction_date', 'inspection_cycle', 'standard_deficiency', 'complaint_deficiency', 'infection_control_inspection_deficiency', 'citation_under_idr', 'citation_under_iidr', 'location', 'processing_date']


<h2>The problem is that this dataset doesn't tell us how many patients a facility has.
    We need to combine two datasets.</h2>

In [310]:
# 1. Trim the Florida dataset to 2024 and 2025, when we have full datasets and Aviata was Aviata (This company keeps changing name every few years)

# fl_health_filtered = fl_df[fl_df['survey_date'].dt.year.isin([2024, 2025])].copy()
# print(f"Florida citations 2024-2025: {len(fl_health_filtered)}")
# print(fl_health_filtered['survey_date'].dt.year.value_counts().sort_index())

In [276]:
fl_citations_filtered = fl_health_filtered[~fl_health_filtered['cms_certification_number_ccn'].isin(aviata_ccns)]
aviata_citations_filtered = fl_health_filtered[fl_health_filtered['cms_certification_number_ccn'].isin(aviata_ccns)]

print(f"Florida (excl. Aviata) citations 2024-2025: {len(fl_citations_filtered)}")
print(f"Aviata citations 2024-2025: {len(aviata_citations_filtered)}")

Florida (excl. Aviata) citations 2024-2025: 5906
Aviata citations 2024-2025: 734


In [417]:
fl_citations_filtered.columns.tolist()

['cms_certification_number_ccn',
 'provider_name',
 'provider_address',
 'citytown',
 'state',
 'zip_code',
 'survey_date',
 'survey_type',
 'deficiency_prefix',
 'deficiency_category',
 'deficiency_tag_number',
 'deficiency_description',
 'scope_severity_code',
 'deficiency_corrected',
 'correction_date',
 'inspection_cycle',
 'standard_deficiency',
 'complaint_deficiency',
 'infection_control_inspection_deficiency',
 'citation_under_idr',
 'citation_under_iidr',
 'location',
 'processing_date']

<h2> Now let's call up the facility dataset to get the number of patients, ratings, etc...</h2>

In [277]:

r = requests.get("https://data.cms.gov/provider-data/api/1/datastore/query/4pq5-n9py/0?conditions[0][property]=state&conditions[0][value]=FL&conditions[0][operator]==&limit=1500")
fl_providers = pd.DataFrame(r.json()['results'])

print(f"CMS FL facilities: {len(fl_providers)}")


# 4pq5-n9py comes from data.cms.gov/provider-data/topics/nursing-homes and then to provider information. It's a code that loads info on providers which we use to get the CCNs.
# 0 means use the first (and main and sometimes only) data source in 4pq5-n9py
# In the documentation, we look at the schemas, and look at conditions (the only one we are interested in)
# The syntax is a query string syntax, which is a common one. It's: conditions[0][property]=state means, I'm defininfg the first conditions (the 0), I'm defining it's property, it's property is state....
# Then on for value and operator. The fact that we need a property, value and operator is spelled out in the documentation

CMS FL facilities: 694


<h2> Now we have two datasets, one on health deficiencies, fl_health, and one for facilities, fl_providers.</h2>
<p> Next, I am goinf to use fl_providers as my base, trim it down to what I need, and add columns based on calculations on fl_health</p>

In [354]:
fl_providers_map = fl_providers[['cms_certification_number_ccn','provider_name','latitude','longitude','average_number_of_residents_per_day','overall_rating']]

fl_providers_map.to_csv('map_florida_nursing_homes.csv', index=False)

In [355]:
fl_providers.columns.tolist()

['cms_certification_number_ccn',
 'provider_name',
 'provider_address',
 'citytown',
 'state',
 'zip_code',
 'telephone_number',
 'provider_ssa_county_code',
 'countyparish',
 'urban',
 'ownership_type',
 'number_of_certified_beds',
 'average_number_of_residents_per_day',
 'average_number_of_residents_per_day_footnote',
 'provider_type',
 'provider_resides_in_hospital',
 'legal_business_name',
 'date_first_approved_to_provide_medicare_and_medicaid_services',
 'chain_name',
 'chain_id',
 'number_of_facilities_in_chain',
 'chain_average_overall_5star_rating',
 'chain_average_health_inspection_rating',
 'chain_average_staffing_rating',
 'chain_average_qm_rating',
 'continuing_care_retirement_community',
 'special_focus_status',
 'abuse_icon',
 'most_recent_health_inspection_more_than_2_years_ago',
 'provider_changed_ownership_in_last_12_months',
 'with_a_resident_and_family_council',
 'automatic_sprinkler_systems_in_all_required_areas',
 'overall_rating',
 'overall_rating_footnote',
 'hea

<h2>Adding a yes and no column for Aviata</h2>
Actually, 0 and 1 because yes and no causes problems in qgis

In [367]:

import numpy as np

fl_providers_map['aviata'] = np.where(fl_providers['provider_name'].str.contains('aviata|aspire', case=False, na=False), '1', '0')
print(fl_providers_map['aviata'].value_counts())
fl_providers_map.to_csv('map_florida_nursing_homes.csv', index=False)
print('-----------')

aviata_ccns = fl_providers_map[fl_providers_map['aviata'] == '1']['cms_certification_number_ccn'].tolist()
print(f"Aviata CCNs: {len(aviata_ccns)}")
print(aviata_ccns)

aviata
0    641
1     53
Name: count, dtype: int64
-----------
Aviata CCNs: 53
['105012', '105132', '105148', '105257', '105258', '105373', '105394', '105413', '105416', '105417', '105419', '105433', '105438', '105440', '105443', '105445', '105452', '105460', '105461', '105465', '105480', '105507', '105531', '105549', '105551', '105558', '105578', '105587', '105588', '105611', '105631', '105632', '105644', '105653', '105663', '105665', '105718', '105764', '105795', '105888', '105895', '105917', '105951', '105952', '105985', '105996', '106000', '106017', '106028', '106029', '106032', '106074', '106116']


<h2>Calculating the average score</h2>

I want to see how many NaN values


In [368]:
na_count = fl_providers_map['overall_rating'].isna().sum()
print(na_count)

3


In [369]:
fl_providers_map[fl_providers_map['overall_rating'].isna()]

,cms_certification_number_ccn,provider_name,latitude,longitude,average_number_of_residents_per_day,overall_rating,aviata
70,105269,GROVES CENTER,27.8936,-81.565,117.3,NaN,0
368,105672,GULF COAST VILLAGE,26.6272,-81.974,86.5,NaN,0
378,105688,AVENTURA AT THE BAY,27.8649,-82.639,196.7,NaN,0


In [370]:
fl_providers_map['overall_rating'] = pd.to_numeric(fl_providers_map['overall_rating'], errors='coerce')

In [371]:
fl_providers_map.nlargest(5, 'overall_rating')
fl_providers_map.nsmallest(5, 'overall_rating')

,cms_certification_number_ccn,provider_name,latitude,longitude,average_number_of_residents_per_day,overall_rating,aviata
7,105029,COMMUNITY CONVALESCENT CENTER,28.0165,-82.146,107.0,1.0,0
21,105117,SOUTH HERITAGE HEALTH & REHABILITATION CENTER,27.7484,-82.643,61.1,1.0,0
36,105155,SARASOTA HEALTH AND REHABILITATION CENTER,27.3198,-82.528,126.8,1.0,0
48,105207,MELBOURNE HEALTHCARE AND REHABILITATION CENTER,28.0867,-80.614,122.0,1.0,0
50,105217,NORTH BEACH HEALTHCARE AND REHABILITATION CENTER,25.9325,-80.156,92.8,1.0,0


<h2>We need to do calculations excluding na values</h2>

Nevermind, Pandas automatically excludes that

In [372]:
fl_providers_map['overall_rating'] = pd.to_numeric(fl_providers_map['overall_rating'], errors='coerce')
fl_providers_map['overall_rating'].mean()

np.float64(3.2821997105643996)

In [373]:
print(fl_providers_map.columns.tolist())

['cms_certification_number_ccn', 'provider_name', 'latitude', 'longitude', 'average_number_of_residents_per_day', 'overall_rating', 'aviata']


In [375]:
aviata_avg_score = fl_providers_map[fl_providers_map['aviata'] == '1']['overall_rating'].mean()
florida_avg_score = fl_providers_map[fl_providers_map['aviata'] == '0']['overall_rating'].mean()

print(f'Aviata average 5-star score is {aviata_avg_score:.2f}')
print(f'Florida average 5-star score is {florida_avg_score:.2f}')

Aviata average 5-star score is 2.32
Florida average 5-star score is 3.36


In [376]:
print(fl_providers_map[fl_providers_map['aviata'] == '1'][['provider_name', 'overall_rating']].to_string())

                           provider_name  overall_rating
5           AVIATA AT THE SEA - PASADENA             2.0
26               AVIATA AT LAKESIDE OAKS             2.0
33              AVIATA AT EMERALD SHORES             4.0
62                 AVIATA AT SAINT LUCIE             1.0
63     AVIATA AT THE SEA - POMPANO BEACH             1.0
133                   Aviata at Sand Key             2.0
148                  AVIATA AT THE PALMS             1.0
162                AVIATA AT BROOKSVILLE             3.0
163                     AVIATA AT BENEVA             1.0
164                    AVIATA AT THE BAY             1.0
165                    AVIATA AT OLDSMAR             1.0
177                AVIATA AT TALLAHASSEE             3.0
182                ASPIRE AT RIDGE HAVEN             3.0
184             AVIATA AT COLONIAL LAKES             2.0
187                     AVIATA AT VENICE             2.0
189           AVIATA AT UNIVERSITY HILLS             4.0
194                  AVIATA AT 

<h2> Average n of citation</h2>
fl_df (deficiency citations) — last 3 years, processing date May 2026, so roughly 2023-2026.
cms_fl (provider info including average_number_of_residents_per_day) — this is a snapshot, not historical. The processing_date is June 2026, so it reflects current resident counts, not a specific year average.

So they don't perfectly align in time — the resident count is current while citations span 3 years. It's the most approximate I could get.

What I am doing is I am going here: https://data.cms.gov/provider-data/archived-data/nursing-homes#2025-annual-files
Downloading all the residents count for each month for 2024 and 2025 (I have already restricted the health citations for those two years)
and combine call them up here to do the work


In [342]:
# Totally vibe coded

import os
import glob

base_path = '/Users/christiancaurla/Desktop/Aviata/Data_analysis/archiveDataSnapshot_2024-25'

all_months = []

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        csvs = glob.glob(os.path.join(folder_path, 'NH_ProviderInfo*.csv'))
        for csv in csvs:
            df = pd.read_csv(csv)
            df['source_file'] = os.path.basename(csv)
            all_months.append(df)

combined = pd.concat(all_months, ignore_index=True)
print(f"Total rows: {len(combined)}")
print(f"Files loaded: {len(all_months)}")
print(combined.columns.tolist())

Total rows: 310940
Files loaded: 21
['CMS Certification Number (CCN)', 'Provider Name', 'Provider Address', 'City/Town', 'State', 'ZIP Code', 'Telephone Number', 'Provider SSA County Code', 'County/Parish', 'Ownership Type', 'Number of Certified Beds', 'Average Number of Residents per Day', 'Average Number of Residents per Day Footnote', 'Provider Type', 'Provider Resides in Hospital', 'Legal Business Name', 'Date First Approved to Provide Medicare and Medicaid Services', 'Affiliated Entity Name', 'Affiliated Entity ID', 'Continuing Care Retirement Community', 'Special Focus Status', 'Abuse Icon', 'Most Recent Health Inspection More Than 2 Years Ago', 'Provider Changed Ownership in Last 12 Months', 'With a Resident and Family Council', 'Automatic Sprinkler Systems in All Required Areas', 'Overall Rating', 'Overall Rating Footnote', 'Health Inspection Rating', 'Health Inspection Rating Footnote', 'QM Rating', 'QM Rating Footnote', 'Long-Stay QM Rating', 'Long-Stay QM Rating Footnote', '

In [346]:
combined['Average Number of Residents per Day'] = pd.to_numeric(combined['Average Number of Residents per Day'], errors='coerce')

aviata_avg_residents = combined[combined['CMS Certification Number (CCN)'].isin(aviata_ccns)]['Average Number of Residents per Day'].mean()
fl_avg_residents = combined[(combined['State'] == 'FL') & (~combined['CMS Certification Number (CCN)'].isin(aviata_ccns))]['Average Number of Residents per Day'].mean()

print(f"Aviata avg residents per day: {aviata_avg_residents:.1f}")
print(f"Florida (excl. Aviata) avg residents per day: {fl_avg_residents:.1f}")

Aviata avg residents per day: 102.9
Florida (excl. Aviata) avg residents per day: 106.5


In [378]:
aviata_facilities = len(aviata_ccns)
fl_facilities = fl_health_filtered['cms_certification_number_ccn'].nunique() - aviata_facilities

aviata_citations_rate = len(aviata_citations_filtered) / aviata_avg_residents / aviata_facilities
fl_citations_rate = len(fl_citations_filtered) / fl_avg_residents / fl_facilities 

print(aviata_citations_rate*100)
print(fl_citations_rate*100)

print(f"Difference: {((aviata_citations_rate - fl_citations_rate) / fl_citations_rate * 100):+.1f}%")


13.462553795667198
9.16577839991501
Difference: +46.9%


In [405]:
fl_providers_map['avg_citations'] = fl_providers_map['tot_citations_2024_25'] / fl_providers_map['avg_residents_2024_25'] *100
fl_providers_map['avg_citations']

0      16.270061
1       8.997429
2       2.495247
3       4.369538
4      22.035677
         ...    
689     3.913894
690     2.831143
691     3.026227
692     6.213385
693    21.310525
Name: avg_citations, Length: 694, dtype: float64

In [406]:
fl_providers_map.head()

,cms_certification_number_ccn,provider_name,latitude,longitude,average_number_of_residents_per_day,overall_rating,aviata,avg_residents_2024_25,tot_citations_2024_25,avg_citations
0,105001,LAKE EUSTIS HEALTHCARE AND REHABILITATION CENTER,28.8474,-81.69,86.7,2.0,0,86.047619,14.0,16.270061
1,105002,BEACH STREET HEALTH AND REHABILITATION CENTER,29.1947,-81.011,88.3,4.0,0,88.914286,8.0,8.997429
2,105005,CORAL GABLES NURSING AND REHABILITATION CENTER,25.7629,-80.309,76.2,5.0,0,80.152381,2.0,2.495247
3,105008,BISCAYNE HEALTH AND REHABILITATION CENTER,25.8915,-80.166,89.2,5.0,0,91.542857,4.0,4.369538
4,105009,GOLFCREST NURSING CENTER,26.0162,-80.142,60.3,3.0,0,63.533333,14.0,22.035677


In [407]:
fl_providers_map.to_csv('/Users/christiancaurla/Desktop/Aviata/Data_analysis/map_florida_nursing_homes.csv', index=False)
print("Exported")

Exported


<h2>This was the tough one. Now let's calculate some more things</h2>

In [379]:
# I don't like that my fl_providers_map has average residents per day for only 2026. We replace it with 2024,2025

# Remove old column:
# fl_providers_map = fl_providers_map.drop(columns=['average_number_of_residents_per_day'], errors='ignore')

# Calculate average from 2024-25 monthly data
combined['Average Number of Residents per Day'] = pd.to_numeric(combined['Average Number of Residents per Day'], errors='coerce')
avg_residents_2024_25 = combined.groupby('CMS Certification Number (CCN)')['Average Number of Residents per Day'].mean().reset_index()
avg_residents_2024_25.columns = ['cms_certification_number_ccn', 'avg_residents_2024_25']

# Merge into fl_providers_map
fl_providers_map = fl_providers_map.merge(avg_residents_2024_25, on='cms_certification_number_ccn', how='left')
print(fl_providers_map[['provider_name', 'avg_residents_2024_25']].head())


                                      provider_name  avg_residents_2024_25
0  LAKE EUSTIS HEALTHCARE AND REHABILITATION CENTER              86.047619
1     BEACH STREET HEALTH AND REHABILITATION CENTER              88.914286
2    CORAL GABLES NURSING AND REHABILITATION CENTER              80.152381
3         BISCAYNE HEALTH AND REHABILITATION CENTER              91.542857
4                          GOLFCREST NURSING CENTER              63.533333


In [381]:
# Now I have a bumch of duplicates!!

# fl_providers_map = fl_providers_map.drop(columns=['citations_2024_25'])
# print(fl_providers_map.columns.tolist())



In [382]:
fl_providers_map.to_csv('map_florida_nursing_homes.csv', index=False)

In [383]:
# I want to know in a separate column how many citations each facility received in 2024 and 2025
# So I will have to add a column 'citations 2024-25' in fl_providers_map

citations_count = fl_health_filtered.groupby('cms_certification_number_ccn').size().reset_index(name='tot_citations_2024_25')
fl_providers_map = fl_providers_map.merge(citations_count, on='cms_certification_number_ccn', how='left')
fl_providers_map['tot_citations_2024_25'] = fl_providers_map['tot_citations_2024_25'].fillna(0)
print(fl_providers_map[['provider_name', 'tot_citations_2024_25']].head())


                                      provider_name  tot_citations_2024_25
0  LAKE EUSTIS HEALTHCARE AND REHABILITATION CENTER                   14.0
1     BEACH STREET HEALTH AND REHABILITATION CENTER                    8.0
2    CORAL GABLES NURSING AND REHABILITATION CENTER                    2.0
3         BISCAYNE HEALTH AND REHABILITATION CENTER                    4.0
4                          GOLFCREST NURSING CENTER                   14.0


In [408]:
aviata_citations_filtered['deficiency_category'].value_counts()

deficiency_category
Quality of Life and Care Deficiencies                         215
Resident Rights Deficiencies                                  113
Resident Assessment and Care Planning Deficiencies            111
Pharmacy Service Deficiencies                                  73
Nutrition and Dietary Deficiencies                             49
Infection Control Deficiencies                                 47
Freedom from Abuse, Neglect, and Exploitation Deficiencies     41
Administration Deficiencies                                    31
Nursing and Physician Services Deficiencies                    27
Environmental Deficiencies                                     27
Name: count, dtype: int64

In [385]:
aviata_citations_filtered['deficiency_category'].value_counts(normalize=True) * 100

deficiency_category
Quality of Life and Care Deficiencies                         29.291553
Resident Rights Deficiencies                                  15.395095
Resident Assessment and Care Planning Deficiencies            15.122616
Pharmacy Service Deficiencies                                  9.945504
Nutrition and Dietary Deficiencies                             6.675749
Infection Control Deficiencies                                 6.403270
Freedom from Abuse, Neglect, and Exploitation Deficiencies     5.585831
Administration Deficiencies                                    4.223433
Nursing and Physician Services Deficiencies                    3.678474
Environmental Deficiencies                                     3.678474
Name: proportion, dtype: float64

In [386]:
aviata_quality_citations = (aviata_citations_filtered['deficiency_category'] == 'Quality of Life and Care Deficiencies').sum()
fl_quality_citations = (fl_citations_filtered['deficiency_category'] == 'Quality of Life and Care Deficiencies').sum()

aviata_quality_citations_rate = aviata_quality_citations / aviata_avg_residents / aviata_facilities
fl_quality_citations_rate = fl_quality_citations / fl_avg_residents / fl_facilities 

print(aviata_quality_citations_rate*100)
print(fl_quality_citations_rate*100)

print(f"Aviata citations per resident: {aviata_quality_citations_rate:.4f}")
print(f"Florida citations per resident: {fl_quality_citations_rate:.4f}")
print(f"Difference: {((aviata_quality_citations_rate - fl_quality_citations_rate) / fl_quality_citations_rate * 100):+.1f}%")


3.943391098185895
2.5560509692956352
Aviata citations per resident: 0.0394
Florida citations per resident: 0.0256
Difference: +54.3%


In [387]:
aviata_residents_citations = (aviata_citations_filtered['deficiency_category'] == 'Resident Rights Deficiencies').sum()
fl_residents_citations = (fl_citations_filtered['deficiency_category'] == 'Resident Rights Deficiencies').sum()

aviata_residents_citations_rate = aviata_residents_citations / aviata_avg_residents / aviata_facilities
fl_residents_citations_rate = fl_residents_citations / fl_avg_residents / fl_facilities 

print(aviata_residents_citations_rate*100)
print(fl_residents_citations_rate*100)

print(f"Aviata citations per resident: {aviata_residents_citations_rate:.4f}")
print(f"Florida citations per resident: {fl_residents_citations_rate:.4f}")
print(f"Difference: {((aviata_residents_citations_rate - fl_residents_citations_rate) / fl_residents_citations_rate * 100):+.1f}%")


2.0725729957907264
1.368814180278537
Aviata citations per resident: 0.0207
Florida citations per resident: 0.0137
Difference: +51.4%


<h2>Let's calculate the severity of those citations wit 'scope_severity_code' </h2>

In [418]:
aviata_ij = aviata_citations_filtered[aviata_citations_filtered['scope_severity_code'].isin(['J','K','L'])]
fl_ij = fl_citations_filtered[fl_citations_filtered['scope_severity_code'].isin(['J','K','L'])]

aviata_ij_pct = len(aviata_ij) / len(aviata_citations_filtered) * 100
fl_ij_pct = len(fl_ij) / len(fl_citations_filtered) * 100
pct_diff = ((aviata_ij_pct - fl_ij_pct) / fl_ij_pct) * 100

print(f"Aviata IJ citations: {len(aviata_ij)} ({aviata_ij_pct:.1f}% of all citations)")
print(f"Florida IJ citations: {len(fl_ij)} ({fl_ij_pct:.1f}% of all citations)")
print(f"Difference: {pct_diff:+.1f}%")

Aviata IJ citations: 44 (6.0% of all citations)
Florida IJ citations: 254 (4.3% of all citations)
Difference: +39.4%


<h2>Do they have more nursing homes than other states in Florida?</h2>
I want to know how many nursing home per citizen florida has compared to average us 
I grab us state populaitons from here: https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_population


In [388]:
import requests
from bs4 import BeautifulSoup

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}

url = 'https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_population'
response = requests.get(url, headers=headers)
print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp


In [389]:
url = 'https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_population'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}

tables = pd.read_html(url, storage_options=headers) #Here I add "storage_options=headers" because I need to tell it, hey get the url but remember this is my header.

pop_us_states = tables[0]

pop_us_states

Unnamed: 0_level_0            State or territory  \
   Unnamed: 0_level_1            State or territory   
0                 1.0                    California   
1                 2.0                         Texas   
2                 3.0                       Florida   
3                 4.0                      New York   
4                 5.0                  Pennsylvania   
5                 6.0                      Illinois   
6                 7.0                          Ohio   
7                 8.0                       Georgia   
8                 9.0                North Carolina   
9                10.0                      Michigan   
10               11.0                    New Jersey   
11               12.0                      Virginia   
12               13.0                    Washington   
13               14.0                       Arizona   
14               15.0                     Tennessee   
15               16.0                 Massachusetts   
16               17.0                       Indiana   
17               18.0                      Missouri   
18               19.0                      Maryland   
19               20.0                      Colorado   
20               21.0                     Wisconsin   
21               22.0                     Minnesota   
22               23.0                South Carolina   
23               24.0                       Alabama   
24               25.0                     Louisiana   
25               26.0                      Kentucky   
26               27.0                        Oregon   
27               28.0                      Oklahoma   
28               29.0                   Connecticut   
29               30.0                          Utah   
30               31.0                        Nevada   
31               32.0                          Iowa   
32               33.0                   Puerto Rico   
33               34.0                      Arkansas   
34               35.0                        Kansas   
35               36.0                   Mississippi   
36               37.0                    New Mexico   
37               38.0                         Idaho   
38               39.0                      Nebraska   
39               40.0                 West Virginia   
40               41.0                        Hawaii   
41               42.0                 New Hampshire   
42               43.0                         Maine   
43               44.0                       Montana   
44               45.0                  Rhode Island   
45               46.0                      Delaware   
46               47.0                  South Dakota   
47               48.0                  North Dakota   
48               49.0                        Alaska   
49               50.0          District of Columbia   
50               51.0                       Vermont   
51               52.0                       Wyoming   
52               53.0                      Guam[11]   
53               54.0       U.S. Virgin Islands[12]   
54               55.0            American Samoa[13]   
55               56.0  Northern Mariana Islands[14]   
56                NaN      Contiguous United States   
57                NaN                 The 50 states   
58                NaN        The 50 states and D.C.   
59                NaN      Total US and territories   

   Census population[8][9][a]               House seats[b]          \
          July 1, 2025 (est.) April 1, 2020          Seats       %   
0                  39355309.0      39538223             52  11.95%   
1                  31709821.0      29145505             38   8.74%   
2                  23462518.0      21538187             28   6.44%   
3                  20002427.0      20201249             26   5.98%   
4                  13059432.0      13002700             17   3.91%   
5                  12719141.0      12812508             17   3.91%   
6                  11900510.0   

In [390]:
pop_us_states.columns.tolist()

[('Unnamed: 0_level_0', 'Unnamed: 0_level_1'),
 ('State or territory', 'State or territory'),
 ('Census population[8][9][a]', 'July 1, 2025 (est.)'),
 ('Census population[8][9][a]', 'April 1, 2020'),
 ('House seats[b]', 'Seats'),
 ('House seats[b]', '%'),
 ('Pop. per elec. vote (2020)[c]', 'Pop. per elec. vote (2020)[c]'),
 ('Pop. per seat (2020)[a]', 'Pop. per seat (2020)[a]'),
 ('% US (2020)', '% US (2020)'),
 ('% EC (2020)', '% EC (2020)')]

In [391]:
pop_by_state = pop_us_states['Census population[8][9][a]']['July 1, 2025 (est.)']

In [392]:
pop_by_state['state'] = pop_us_states['State or territory']['State or territory']

In [393]:
us_state_pop = pd.DataFrame({
    'state': pop_us_states['State or territory']['State or territory'],
    'population_2025': pop_us_states['Census population[8][9][a]']['July 1, 2025 (est.)']
})
print(us_state_pop.head())

          state  population_2025
0    California       39355309.0
1         Texas       31709821.0
2       Florida       23462518.0
3      New York       20002427.0
4  Pennsylvania       13059432.0


In [394]:
state_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC',
    'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC'
}

us_state_pop['state_abbrev'] = us_state_pop['state'].map(state_abbrev)
print(us_state_pop.head())

          state  population_2025 state_abbrev
0    California       39355309.0           CA
1         Texas       31709821.0           TX
2       Florida       23462518.0           FL
3      New York       20002427.0           NY
4  Pennsylvania       13059432.0           PA


In [395]:
# Vibe coded this

providers_by_state = []

for abbrev in us_state_pop['state_abbrev'].dropna():
    r = requests.get(
        f"https://data.cms.gov/provider-data/api/1/datastore/query/4pq5-n9py/0"
        f"?conditions[0][property]=state&conditions[0][value]={abbrev}&conditions[0][operator]==&limit=1500"
    )
    count = r.json().get('count', 0)
    providers_by_state.append({'state_abbrev': abbrev, 'nursing_homes': count})
    print(f"{abbrev}: {count}")

providers_by_state_df = pd.DataFrame(providers_by_state)
print(providers_by_state_df.head())

CA: 1164
TX: 1176
FL: 694
NY: 596
PA: 656
IL: 667
OH: 921
GA: 356
NC: 420
MI: 423
NJ: 348
VA: 289
WA: 193
AZ: 140
TN: 303
MA: 341
IN: 507
MO: 487
MD: 221
CO: 210
WI: 323
MN: 338
SC: 187
AL: 224
LA: 266
KY: 268
OR: 128
OK: 283
CT: 191
UT: 97
NV: 66
IA: 389
AR: 221
KS: 296
MS: 202
NM: 68
ID: 80
NE: 179
WV: 123
HI: 42
NH: 73
ME: 78
MT: 61
RI: 72
DE: 44
SD: 96
ND: 72
AK: 20
DC: 17
VT: 33
WY: 36
  state_abbrev  nursing_homes
0           CA           1164
1           TX           1176
2           FL            694
3           NY            596
4           PA            656


In [396]:
providers_by_state_df['population_2025'] = us_state_pop['population_2025']
providers_by_state_df.head()

,state_abbrev,nursing_homes,population_2025
0,CA,1164,39355309.0
1,TX,1176,31709821.0
2,FL,694,23462518.0
3,NY,596,20002427.0
4,PA,656,13059432.0


In [397]:
providers_by_state_df['facilities_per_resident'] = providers_by_state_df['population_2025'] / providers_by_state_df['nursing_homes']
providers_by_state_df.head()

,state_abbrev,nursing_homes,population_2025,facilities_per_resident
0,CA,1164,39355309.0,33810.402921
1,TX,1176,31709821.0,26964.133503
2,FL,694,23462518.0,33807.662824
3,NY,596,20002427.0,33561.119128
4,PA,656,13059432.0,19907.670732


In [398]:
print(providers_by_state_df.sort_values(by='facilities_per_resident'))

   state_abbrev  nursing_homes  population_2025  facilities_per_resident
31           IA            389        3238387.0              8324.902314
33           KS            296        3114791.0             10522.942568
45           SD             96        1059952.0             11041.166667
37           NE            179        2029733.0             11339.290503
17           MO            487        6270541.0             12875.854209
6            OH            921       11900510.0             12921.292074
46           ND             72         935094.0             12987.416667
16           IN            507        6973333.0             13754.108481
32           AR            221        3184835.0             14411.018100
27           OK            283        4123288.0             14569.922261
34           MS            202        2977220.0             14738.712871
43           RI             72        1144694.0             15898.527778
38           WV            123        2018006.0    

<h2> Well, this was a bit of a waste of time</h2>

In [399]:
fl_providers_map.to_csv('map_florida_nursing_homes.csv', index=False)